# W1 Lab — First Model Calls and Prompting Fundamentals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week01/W1_lab_setup.ipynb)

**Goal.** Call a language model from code, know what one call does and does not do,
and control it through the system prompt — precisely enough that the final task's
answers come back as clean JSON, checked by machine.

Why this comes first: the notes (Ch. 1) define an agent as code that acts on model
output, so everything starts with making the call and making its output usable by code.

The path: setup (1) → one call taken apart: roles, statelessness, temperature (2) →
why "just ask for JSON" is not enough (3) → the core task + two exercises (4) → cost
and completion (5).

*Runtime:* Google Colab, top-to-bottom, ~80 minutes. Cells marked ✍️ ask for your own
writing — a fill-in or a written prediction.

*Sources:* lab format adapted from DeepLearning.AI, *Agentic AI* (Andrew Ng); prompting
exercises draw on Anthropic's *Prompt Engineering Interactive Tutorial*.


## 1. Setup

### 1.1 Installation

`aisuite` exposes multiple providers (OpenAI, Anthropic) behind one interface, so the
same code runs whichever provider your key belongs to.

*Do:* run the cell (~30 seconds, once per session).


In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

An **API key** = the secret string that identifies your account to the provider and
bills usage to it (issuing steps: the API Setup guide on the course site). The key is
yours; do not share the notebook with the key still inside.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell. Nothing prints.


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 One Call, Written Out

A **message** = one turn of the exchange: a dict with a `role` and a `content` string.
`user` carries the request. A call sends a list of messages and gets back a response
object; the reply text sits at `response.choices[0].message.content`.

*Do:* run the cell. Compare the `messages` list you sent with the reply you got back.


In [ ]:
import aisuite

client = aisuite.Client()

messages = [{"role": "user", "content": "Reply with exactly: ready"}]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)

print("messages sent:", messages)
print("reply text   :", response.choices[0].message.content)


Any error here is a setup problem, not a code problem — recheck the API Setup guide
before continuing.


## 2. Anatomy of a Model Call

### 2.1 Messages and roles

Section 1 sent a list holding one `user` message. A message can also carry `system`,
for standing instructions, or `assistant`, for the model's own previous replies — a
call receives a list of these and nothing else.

The cell below asks the same question twice, each call written out the way Section 1
wrote it. The two lists differ by one dict: a `system` message at the front of the
second one.

*Do:* run the cell. Read the two printed lists first, then the two answers.

In [ ]:
QUESTION = "What is an AI agent?"

plain_messages = [
    {"role": "user", "content": QUESTION},
]

instructed_messages = [
    {"role": "system", "content": "Answer in one sentence, for a graduate ML audience."},
    {"role": "user", "content": QUESTION},
]

print("PLAIN LIST     :", plain_messages)
print("INSTRUCTED LIST:", instructed_messages)

plain = client.chat.completions.create(model=MODEL, messages=plain_messages, temperature=0.0)
instructed = client.chat.completions.create(model=MODEL, messages=instructed_messages, temperature=0.0)

print("\nPLAIN:\n", plain.choices[0].message.content)
print("\nINSTRUCTED:\n", instructed.choices[0].message.content)

The instruction is a separate dict at the front of the list, not text prepended to the
question. An application fixes that dict once and varies only the `user` entry —
Section 4 turns this into the task.


### 2.2 Statelessness

An API call is **stateless** = the model sees only the message list inside that one
call; the provider keeps no conversation state between calls. The experiment tells one
call a fact and asks the next call to repeat it.

*Do:* run the cell and read the second answer.


In [ ]:
first_messages = [
    {"role": "user", "content": "My research topic is maritime logistics optimization. "
                                "Acknowledge in five words."},
]
first = client.chat.completions.create(model=MODEL, messages=first_messages, temperature=0.0)

second_messages = [
    {"role": "user", "content": "What is my research topic?"},
]
second = client.chat.completions.create(model=MODEL, messages=second_messages, temperature=0.0)

print("FIRST :", first.choices[0].message.content)
print("SECOND:", second.choices[0].message.content)

The second call cannot answer, because the first call's content never reached it.
Conversation is an application-side construct: the client resends the accumulated
history with every call.

In [ ]:
conversation = [
    {"role": "user", "content": "My research topic is maritime logistics optimization. "
                                "Acknowledge in five words."},
]
response = client.chat.completions.create(model=MODEL, messages=conversation, temperature=0.0)
conversation.append({"role": "assistant", "content": response.choices[0].message.content})
conversation.append({"role": "user", "content": "What is my research topic?"})

response = client.chat.completions.create(model=MODEL, messages=conversation, temperature=0.0)
print(response.choices[0].message.content)

With the history resent, the model answers. Everything a model appears to remember is
text that some code placed into the message list.

### Exercise — one more turn ✍️

Add a third user turn to `conversation` (any follow-up about the topic), resend the
full history with another `create` call, and print the reply. The pattern is the
`append` and `create` lines above.

In [ ]:
### FILL IN (START) ###
# conversation.append({"role": "user", "content": "..."})
# response = client.chat.completions.create(model=MODEL, messages=conversation, temperature=0.0)
# print(response.choices[0].message.content)
### FILL IN (END) ###

### 2.3 Temperature

**Temperature** = the sampling parameter that scales the spread of the next-token
distribution: 0 collapses sampling toward the most probable token, higher values admit
less probable ones. The cell repeats one open question twice at each setting.

*Do:* run the cell and compare within each pair.


In [ ]:
probe_messages = [
    {"role": "user", "content": "Name one promising research direction for LLM agents, "
                                "in one sentence."},
]

for temperature in [0.0, 0.0, 1.0, 1.0]:
    response = client.chat.completions.create(
        model=MODEL, messages=probe_messages, temperature=temperature)
    print(f"t={temperature} :", response.choices[0].message.content)

The two `t=0.0` runs are (near-)identical; the `t=1.0` runs differ. Deterministic
settings suit checking and grading; sampled diversity is itself useful and W2 builds a
technique on it (self-consistency).

## 3. Format Control

Code must act on the output, and code can act only on what it can parse.
**Structured output** = model output constrained to a machine-parseable format (here
JSON). Models often wrap JSON in Markdown code fences, so the parser below strips them
before `json.loads`.

*Do:* run the cell, more than once if you like. A parsed dict means the loose request
happened to work this time; `PARSE FAILED` means it did not — nothing in the request
guarantees either.


In [ ]:
import json
import re

def parse_json_output(text):
    """Model output -> parsed JSON object; raises ValueError if unparseable."""
    stripped = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        return json.loads(stripped)
    except json.JSONDecodeError as exc:
        raise ValueError(f"unparseable model output: {text[:120]!r}") from exc

messages = [
    {"role": "user", "content": "Give the title and year of the Chain-of-Thought paper as JSON."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
attempt = response.choices[0].message.content

print(attempt)
try:
    print(parse_json_output(attempt))
except ValueError as exc:
    print("PARSE FAILED:", exc)

A loose request sometimes yields prose around the object. The fix is not post-hoc
string surgery; it is an instruction that pins the format exactly — Section 4.


## 4. Task — Answers as JSON ✍️ (core)

One system prompt, stated precisely enough that its effect passes a machine check.

An example first: the cell below solves the same problem for a smaller schema — a
sentiment classifier that must return `{"label": ..., "confidence": ...}`. Read the
prompt before running it: it names every key, states the closed vocabulary for
`label`, and forbids everything else.

In the message list, the schema lives entirely in the `system`
turn, and the `user` turn carries only the text to classify.

*Do:* run the cell and read the raw reply next to its parsed form.


In [ ]:
EXAMPLE_SYSTEM = (
    "You are a sentiment classifier. For every user text, return ONLY a raw JSON "
    "object — no prose, no Markdown code fences — with exactly these keys and no "
    "others:\n"
    '  "label": exactly one of "positive", "neutral", "negative";\n'
    '  "confidence": a number between 0 and 1.\n'
    "Never add keys, comments, or explanations outside the JSON object."
)
EXAMPLE_USER = "Classify the sentiment of this text.\n\nText: {text}"

messages = [
    {"role": "system", "content": EXAMPLE_SYSTEM},
    {"role": "user", "content": EXAMPLE_USER.format(text="The lab finally makes sense to me.")},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
reply = response.choices[0].message.content

print("raw reply:", reply)
print("parsed:   ", parse_json_output(reply))

### 4.1 Writing `JSON_SYSTEM` ✍️

Write `JSON_SYSTEM` so that the answer to any question comes back as raw JSON with
exactly these keys:

| key | content |
|---|---|
| `topic` | short label for the subject area |
| `answer` | one-sentence direct answer |
| `difficulty` | one of `"intro"`, `"intermediate"`, `"advanced"` |

Follow the pattern of `EXAMPLE_SYSTEM`: name every key, state the closed vocabulary
for `difficulty`, forbid everything else — no prose around the object, no code fences,
no extra keys.

*Do:* fill in `JSON_SYSTEM`, run the check cell, iterate until both questions read
`PASS`. Typical first failures: code fences, or extra keys the model volunteers —
both disappear once forbidden explicitly.


In [ ]:
### FILL IN (START) ###
JSON_SYSTEM = (
    ""
)
### FILL IN (END) ###

In [ ]:
QUESTIONS = [
    "How can I use an LLM to answer questions over our lab's PDF reports?",
    "Why does asking the model to think step by step improve accuracy?",
]
REQUIRED_KEYS = {"topic", "answer", "difficulty"}

n_passed = 0
for question in QUESTIONS:
    messages = [
        {"role": "system", "content": JSON_SYSTEM},
        {"role": "user", "content": question},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    try:
        ok = set(parse_json_output(output)) == REQUIRED_KEYS
    except ValueError:
        ok = False
    n_passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {question[:55]}...")
    if not ok:
        print("      output began:", repr(output[:90]))

print("ALL PASS" if n_passed == len(QUESTIONS) else "KEEP ITERATING")

### 4.2 Exercise — temperature vs. format ✍️

Prediction, written down before running: at `t=1.0`, does the answer still parse as
valid JSON with the three keys on every run?

*Do:* run the cell and compare with your prediction.

In [ ]:
for run in range(3):
    messages = [
        {"role": "system", "content": JSON_SYSTEM},
        {"role": "user", "content": QUESTIONS[0]},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0)
    try:
        ok = set(parse_json_output(response.choices[0].message.content)) == REQUIRED_KEYS
    except ValueError:
        ok = False
    print(f"run {run}: {'PASS' if ok else 'FAIL'}")

### 4.3 Exercise — provider-enforced JSON mode

OpenAI can enforce JSON at the API level: `response_format={"type": "json_object"}`
constrains decoding so the reply always parses. It guarantees parseable JSON, not your
three keys — those still come from `JSON_SYSTEM` — and it is OpenAI-specific. (The API
requires the word "JSON" somewhere in the messages; `JSON_SYSTEM` already satisfies
this.)

Prediction: with the parameter added, can the reply still violate the three-key schema
even though it always parses?

*Do:* run the cell.


In [ ]:
messages = [
    {"role": "system", "content": JSON_SYSTEM},
    {"role": "user", "content": QUESTIONS[0]},
]
response = client.chat.completions.create(
    model=MODEL, messages=messages, temperature=0.0,
    response_format={"type": "json_object"})
output = response.choices[0].message.content

print(output)
print(parse_json_output(output))

## 5. Cost and Completion

Every response object carries a `usage` field: the token counts the provider just
billed for that call. The cell reads it off the JSON-mode response from 4.3 and
prices that one call at gpt-4o-mini list prices.

In [ ]:
PRICE_PER_M_PROMPT = 0.15        # USD per 1M input tokens, gpt-4o-mini
PRICE_PER_M_COMPLETION = 0.60    # USD per 1M output tokens, gpt-4o-mini

usage = response.usage
cost = (usage.prompt_tokens * PRICE_PER_M_PROMPT
        + usage.completion_tokens * PRICE_PER_M_COMPLETION) / 1_000_000
print(f"prompt tokens: {usage.prompt_tokens}   completion tokens: {usage.completion_tokens}")
print(f"cost of that one call: ${cost:.6f}")

Multiplied over this session's few dozen calls, the lab costs on the order of a cent.
Call and token counts, not dollars, become the binding constraint once loops multiply
calls (notes Ch. 12).

### Completion check

All rows must read `PASS` before submission; grading checks these structural facts,
never prose quality.

In [ ]:
completion = {
    "JSON-mode reply parsed":             isinstance(output, str) and len(output) > 0,
    "JSON_SYSTEM written (>= 40 chars)":  len(JSON_SYSTEM.strip()) >= 40,
    "both JSON questions PASS":           n_passed == len(QUESTIONS),
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")

---

W2 works on the content of the prompt rather than its format: chain-of-thought,
worked-example exemplars, and sampling-based self-consistency, measured on a small
code-graded evalset. After a week of writing every call out by hand, W2 also
introduces the `ask` helper that wraps the repeated lines. Reference answers for
this lab: `labs/checkpoints/week01/solution.py`, published after the session.